# Module 2: Data Cleaning & Transformation

This notebook takes the raw merged dataset from Module 1 (`university_raw_data.csv`) and cleans it up so it's ready for Tableau.
- Remove duplicates.
- Standardize university names.
- Standardize country names.
- Normalize ranking metrics.
- Create Tableau-ready datasets.

## Step 1: Load the raw file

In [1]:
import pandas as pd

df = pd.read_csv("university_raw_data.csv", header=1, low_memory=False)
df = df.drop(columns=["Unnamed: 0"])  # duplicate of the row number, not useful

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded: 22117 rows x 38 columns


,university_id,university_name,year,world_rank,national_rank,overall_score,country,region,city,university_type,...,degree_level,undergraduate_count,postgraduate_count,country_avg_rank,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio
0,1,ADA University,2026,1300.5,9.0,40.600000,Azerbaijan,Asia,Azerbaijan,Public,...,Undergraduate & Postgraduate,2461,2399,963.00,9,688.0,38.93,17.37,1.61,18.11
1,2,AGH University of Krakow,2016,723.0,4.0,18.525519,Poland,Europe,Poland,Public,...,Undergraduate & Postgraduate,26756,8813,681.14,7,507.0,20.20,19.84,21.72,2.57
2,2,AGH University of Krakow,2017,711.0,7.0,22.590000,Poland,Europe,Poland,Public,...,Undergraduate & Postgraduate,22164,11080,713.75,12,366.0,23.06,22.35,22.97,275.83
3,2,AGH University of Krakow,2018,789.0,4.0,22.365000,Poland,Europe,Poland,Public,...,Undergraduate & Postgraduate,21842,11088,847.21,17,415.5,19.60,19.74,19.77,332.45
4,2,AGH University of Krakow,2019,983.0,15.0,19.337500,Poland,Europe,Poland,Public,...,Undergraduate & Postgraduate,24491,6761,883.90,20,394.0,23.08,23.16,23.36,699.70


## Step 2: Standardize university names

In [2]:
df["university_name"] = df["university_name"].str.strip().str.replace(r"\s+", " ", regex=True)

name_key = df["university_name"].str.lower()
canonical_name = df.groupby(name_key)["university_name"].agg(lambda s: s.value_counts().idxmax())
df["university_name"] = name_key.map(canonical_name)

# Re-number university_id now that name variants are merged, so the same
# university always gets the same ID across every year
df["university_id"] = df["university_name"].astype("category").cat.codes + 1

df[["university_id", "university_name"]].head()

,university_id,university_name
0,1,ADA University
1,2,AGH University of Krakow
2,2,AGH University of Krakow
3,2,AGH University of Krakow
4,2,AGH University of Krakow


## Step 3: Standardize country names and fix regions
Several countries appear under multiple spellings - these are mapped to one consistent name.

In [3]:
COUNTRY_ALIASES = {
    "China (Mainland)": "China",
    "Brunei Darussalam": "Brunei",
    "Czechia": "Czech Republic",
    "Hong Kong SAR": "Hong Kong",
    "Hong Kong SAR, China": "Hong Kong",
    "Iran (Islamic Republic of)": "Iran",
    "Iran, Islamic Republic of": "Iran",
    "Macao SAR, China": "Macao",
    "Macau SAR": "Macao",
    "Palestinian Territory, Occupied": "Palestine",
    "Republic of Korea": "South Korea",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "T\u00fcrkiye": "Turkey",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Viet Nam": "Vietnam",
}

REGION_FIXES = {
    "United States": "North America", "Canada": "North America", "Puerto Rico": "North America",
    "Mexico": "Latin America", "Brazil": "Latin America", "Colombia": "Latin America",
    "Chile": "Latin America", "Argentina": "Latin America", "Ecuador": "Latin America",
    "Peru": "Latin America", "Costa Rica": "Latin America", "Venezuela": "Latin America",
    "Dominican Republic": "Latin America", "Uruguay": "Latin America", "Cuba": "Latin America",
    "Panama": "Latin America", "Guatemala": "Latin America", "Paraguay": "Latin America",
    "Bolivia": "Latin America", "Honduras": "Latin America", "Jamaica": "Latin America",
    "Algeria": "Africa", "Tanzania": "Africa", "Botswana": "Africa", "Mozambique": "Africa",
    "Mauritius": "Africa", "Namibia": "Africa", "Zambia": "Africa", "Zimbabwe": "Africa",
    "Rwanda": "Africa", "Senegal": "Africa", "Democratic Republic of the Congo": "Africa",
    "Nepal": "Asia", "Mongolia": "Asia", "Yemen": "Asia", "Northern Cyprus": "Europe",
    "Montenegro": "Europe", "North Macedonia": "Europe", "Kosovo": "Europe",
    "Fiji": "Oceania",
}

df["country"] = df["country"].str.strip().replace(COUNTRY_ALIASES)
df["city"] = df["city"].str.strip()

vague_region = df["region"].isin(["Other", "Americas", "Not Classified"])
df.loc[vague_region, "region"] = df.loc[vague_region, "country"].map(REGION_FIXES)

print(sorted(df["region"].unique()))

['Africa', 'Asia', 'Europe', 'Latin America', 'North America', 'Oceania']


## Step 4: Remove duplicates

In [4]:
before = len(df)
df["_missing_count"] = df.isna().sum(axis=1)
df = df.sort_values(["_missing_count", "world_rank"]).drop_duplicates(
    subset=["university_name", "year"], keep="first"
)
df = df.drop(columns=["_missing_count"]).sort_index()

print(f"Removed {before - len(df)} duplicate university/year rows")
print(f"Remaining duplicates: {df.duplicated(subset=['university_name', 'year']).sum()}")

Removed 102 duplicate university/year rows
Remaining duplicates: 0


## Step 5: Normalize ranking / performance metrics

In [5]:
invalid_ratio = (df["international_student_ratio"] < 0) | (df["international_student_ratio"] > 100)
df.loc[invalid_ratio, "international_student_ratio"] = df.loc[invalid_ratio, "country_avg_international_ratio"]
df["international_student_ratio"] = df["international_student_ratio"].clip(0, 100)

df["gender_ratio"] = (
    df["female_percentage"].round().astype(int).astype(str) + ":" +
    df["male_percentage"].round().astype(int).astype(str)
)

df["university_type"] = df["university_type"].replace({"C": "Not Specified"})

print("international_student_ratio range:", df["international_student_ratio"].min(), "-", df["international_student_ratio"].max())
print(df["university_type"].value_counts())

international_student_ratio range: 0.0 - 100.0
university_type
Public                    16581
Private                    4934
Private not for Profit      312
Not Specified               130
Private for Profit           58
Name: count, dtype: int64


## Step 6: Completeness check

In [6]:
completeness = 100 - df.isna().mean() * 100
print("Column completeness (%):")
print(completeness.round(1).sort_values())
print(f"\nOverall completeness: {completeness.mean():.2f}%")

Column completeness (%):
world_rank                          98.9
national_rank                       98.9
university_name                    100.0
university_id                      100.0
year                               100.0
overall_score                      100.0
country                            100.0
region                             100.0
city                               100.0
university_type                    100.0
academic_reputation_score          100.0
employer_reputation_score          100.0
citations_score                    100.0
publications_count                 100.0
citations_count                    100.0
citations_per_faculty              100.0
h_index                            100.0
research_output_score              100.0
research_productivity_index        100.0
subject_field                      100.0
total_students                     100.0
international_students_count       100.0
international_student_ratio        100.0
faculty_count                   

In [7]:
df.to_csv("university_cleaned.csv", index=False)
print(f"Saved university_cleaned.csv - {df.shape[0]} rows x {df.shape[1]} columns")

Saved university_cleaned.csv - 22015 rows x 38 columns
